## 1. Setup: Imports, Data Loading & Preprocessing

Load the dataset, clean it, merge text fields, encode labels, and split into train/val/test.

In [ ]:
import re
import copy
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import seaborn as sns
import nltk
from nltk.corpus import stopwords
from transformers import BertTokenizer, BertModel
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import f1_score, classification_report, roc_curve, auc, multilabel_confusion_matrix
from collections import Counter


df = pd.read_csv("/kaggle/input/datasets/julian3833/jigsaw-toxic-comment-classification-challenge/train.csv")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

TEXT_COL = "comment_text"
LABEL_COLS = ["toxic", "severe_toxic", "obscene", "threat", "insult", "identity_hate"]

df = df.dropna(subset=[TEXT_COL] + LABEL_COLS)
df = df.drop_duplicates(subset=[TEXT_COL]).reset_index(drop=True)
df["text"] = df[TEXT_COL].astype(str)

df[LABEL_COLS] = df[LABEL_COLS].astype(int)
print(df.shape)
df.head()

class_names = LABEL_COLS
num_classes = len(class_names)

# --- ADD THESE NEW LINES ---
# 1. Split original train data ONLY into Train and Validation (85% / 15%)
train_df, val_df = train_test_split(df, test_size=0.15, stratify=df["toxic"], random_state=42)

# 2. Load and merge separate official Test data
DATA_DIR = "/kaggle/input/datasets/julian3833/jigsaw-toxic-comment-classification-challenge/"
test_texts_df = pd.read_csv(DATA_DIR + "test.csv")
test_labels_df = pd.read_csv(DATA_DIR + "test_labels.csv")

# Merge texts and labels on competition 'id'
test_df = pd.merge(test_texts_df, test_labels_df, on="id")

# CRITICAL FILTER: Remove rows where labels are -1 (unscored test samples)
test_df = test_df[test_df[LABEL_COLS[0]] != -1].reset_index(drop=True)

# Clean and format test text fields to match down-stream CustomDataset requirements
test_df = test_df.dropna(subset=[TEXT_COL] + LABEL_COLS)
test_df["text"] = test_df[TEXT_COL].astype(str)
test_df[LABEL_COLS] = test_df[LABEL_COLS].astype(int)

print(f"Train shape: {train_df.shape} | Val shape: {val_df.shape} | External Test shape: {test_df.shape}")

## 2. Tokenizer, Vocabulary & DataLoaders

Custom regex tokenizer, vocab built from training text, dataset/collate logic, and the three DataLoaders.

In [ ]:
# Initialize the pre-trained BERT tokenizer
bert_tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

class BertDataset(Dataset):
    def __init__(self, df, max_len=128):
        self.texts = df["text"].values
        self.labels = df[LABEL_COLS].values.astype("float32")
        self.max_len = max_len
        
    def __len__(self):
        return len(self.texts)
        
    def __getitem__(self, idx):
        text = str(self.texts[idx])
        # BERT tokenizer handles tokenization, UNK mapping, and max_len truncation automatically
        inputs = bert_tokenizer(
            text,
            max_length=self.max_len,
            padding=False,         # Padding will be handled dynamically in the collate function
            truncation=True,
            return_tensors=None
        )
        return inputs["input_ids"], self.labels[idx]

def bert_collate_fn(batch):
    # Sort batch by sequence length for packed sequences
    batch.sort(key=lambda x: len(x[0]), reverse=True)
    sequences, labels = zip(*batch)
    
    lengths = torch.tensor([len(seq) for seq in sequences], dtype=torch.long)
    lengths = torch.clamp(lengths, min=1)
    
    max_len = lengths.max().item()
    
    # BERT uses token ID 0 for [PAD] in bert-base-uncased
    padded = torch.zeros(len(sequences), max_len, dtype=torch.long) 
    for i, seq in enumerate(sequences):
        if len(seq) == 0: seq = [101] # 101 is [CLS] token placeholder if empty
        padded[i, :len(seq)] = torch.tensor(seq, dtype=torch.long)
        
    labels = torch.tensor(np.array(labels), dtype=torch.float32)
    return padded, lengths, labels

# Recreate dataset instances
train_custom = BertDataset(train_df, max_len=128)
val_custom = BertDataset(val_df, max_len=128)
test_custom = BertDataset(test_df, max_len=128)

train_loader = DataLoader(train_custom, batch_size=32, shuffle=True, collate_fn=bert_collate_fn)
val_loader = DataLoader(val_custom, batch_size=32, shuffle=False, collate_fn=bert_collate_fn)
test_loader = DataLoader(test_custom, batch_size=32, shuffle=False, collate_fn=bert_collate_fn)

# Note: Since BERT manages its vocabulary internally, we no longer need to print custom UNK rates.

## 3. Model Definition : BiLSTM

In [ ]:
class BertBiLSTM(nn.Module):
    def __init__(self, hidden_dim=256, output_dim=6, freeze_bert_layers=8):
        super().__init__()
        self.bert = BertModel.from_pretrained('bert-base-uncased')

        for param in self.bert.embeddings.parameters():
            param.requires_grad = False
        for layer in self.bert.encoder.layer[:freeze_bert_layers]:
            for param in layer.parameters():
                param.requires_grad = False

        self.lstm = nn.LSTM(768, hidden_dim, num_layers=2, batch_first=True, bidirectional=True)
        self.dropout = nn.Dropout(0.5)
        self.attn = nn.Linear(hidden_dim * 2, 1)
        self.fc = nn.Linear(hidden_dim * 2, output_dim)

    def forward(self, x, lengths):
        attention_mask = (x != 0).long()
        bert_out = self.bert(input_ids=x, attention_mask=attention_mask).last_hidden_state

        packed = pack_padded_sequence(bert_out, lengths.cpu(), batch_first=True, enforce_sorted=True)
        packed_out, _ = self.lstm(packed)
        out, _ = pad_packed_sequence(packed_out, batch_first=True)

        mask = (x[:, :out.size(1)] != 0)
        attn_scores = self.attn(out).squeeze(-1)
        attn_scores = attn_scores.masked_fill(~mask, float("-inf"))
        attn_weights = torch.softmax(attn_scores, dim=1).unsqueeze(-1)
        final = (out * attn_weights).sum(dim=1)
        return self.fc(self.dropout(final))

## 4. Training Pipeline & Run Training

Trains for 30 epochs, tracks loss/val F1, keeps best checkpoint. Then trains both models.

In [ ]:
def train_pipeline(model_class, name):
    # Initialize the BERT BiLSTM
    model = model_class(output_dim=num_classes).to(device)
    
    # Set up parameter groups for differential learning rates
    bert_params = [p for n, p in model.named_parameters() if n.startswith("bert.") and p.requires_grad]
    head_params = [p for n, p in model.named_parameters() if not n.startswith("bert.") and p.requires_grad]
    
    optimizer = torch.optim.AdamW(
        [
            {"params": bert_params, "lr": 2e-5},
            {"params": head_params, "lr": 1e-3},
        ],
        weight_decay=1e-4,
    )
    
    # Tracking variables for best performance and metrics history
    best_f1 = -1.0
    best_state = None
    history = {"train_loss": [], "val_f1": []}

    # Early Stopping Configurations
    patience = 5  # Stop training if validation F1 doesn't improve for 5 consecutive epochs
    patience_counter = 0

    print(f"\n--- Started Training: {name} ---")
    
    for epoch in range(30):
        # 1. Training Phase
        model.train()
        total_loss, count = 0, 0
        for x, lengths, y in train_loader:
            x, lengths, y = x.to(device), lengths.to(device), y.to(device)
            optimizer.zero_grad()
            loss = criterion(model(x, lengths), y)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            optimizer.step()
            total_loss += loss.item() * x.size(0)
            count += x.size(0)
            
        train_epoch_loss = total_loss / count
        history["train_loss"].append(train_epoch_loss)
            
        # 2. Evaluation Phase
        model.eval()
        preds, targets = [], []
        with torch.no_grad():
            for x, lengths, y in val_loader:
                x, lengths = x.to(device), lengths.to(device)
                out = (torch.sigmoid(model(x, lengths)) > 0.5).int().cpu().numpy()
                preds.extend(out)
                targets.extend(y.numpy().astype(int))
                
        val_f1 = f1_score(targets, preds, average='weighted', zero_division=0)
        history["val_f1"].append(val_f1)
        
        print(f"Epoch {epoch+1:02d}/30 | Train Loss: {train_epoch_loss:.4f} | Val F1: {val_f1:.4f}")
        
        # 3. Check Performance and Handle Early Stopping
        if val_f1 > best_f1:
            best_f1 = val_f1
            best_state = copy.deepcopy(model.state_dict())
            patience_counter = 0  # Reset counter since improvement occurred
        else:
            patience_counter += 1  # No improvement, increment counter
            
        if patience_counter >= patience:
            print(f" Early stopping triggered! No validation F1 improvement for {patience} epochs.")
            break
            
    # Load the best weights discovered before returning
    if best_state is not None:
        model.load_state_dict(best_state)
    else:
        # Fallback safeguard in case no epoch managed to exceed initial best_f1 (-1.0)
        model.load_state_dict(copy.deepcopy(model.state_dict()))
        
    print(f"--- Finished {name} | Best Val F1: {best_f1:.4f} ---")
    return model, history

# Calculate class weights from training dataframe to handle the severe label imbalance
neg_counts = len(train_df) - train_df[LABEL_COLS].sum(axis=0).values
pos_counts = train_df[LABEL_COLS].sum(axis=0).values

# Using the damped square-root multiplier to balance precision and recall
pos_weights = torch.tensor(np.sqrt(neg_counts / pos_counts), dtype=torch.float).to(device)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weights)

print("\nTraining BERT-Initialized Bidirectional BiLSTM...")
model_bilstm, hist_bilstm = train_pipeline(BertBiLSTM, "BertBiLSTM")

## 5. Evaluation & Classification Reports

In [ ]:
def evaluate_model(model, loader):
    model.eval()
    preds, targets, probs = [], [], []
    with torch.no_grad():
        for x, lengths, y in loader:
            x, lengths = x.to(device), lengths.to(device)
            logits = model(x, lengths)
            prob = torch.sigmoid(logits)
            probs.extend(prob.cpu().numpy())
            preds.extend((prob > 0.5).int().cpu().numpy())
            targets.extend(y.numpy().astype(int))
    return np.array(targets), np.array(preds), np.array(probs)
    
# --- ADD THESE NEW LINES ---
# 1. Final evaluation pass on validation set to find thresholds
val_true, _, val_probs = evaluate_model(model_bilstm, val_loader)

# Default 0.5 threshold validation reference
default_preds_val = (val_probs >= 0.5).astype(int)
print(f"Default threshold (0.5) Validation Weighted F1: {f1_score(val_true, default_preds_val, average='weighted', zero_division=0):.4f}")

# 2. Per-label grid-search threshold tuning strictly using the Validation set
thresholds = np.arange(0.05, 0.95, 0.05)
best_thresholds = np.zeros(num_classes)

print("\n--- Optimizing Thresholds on Validation Set ---")
for i, label in enumerate(class_names):
    best_f1, best_t = 0.0, 0.5
    for t in thresholds:
        preds_i = (val_probs[:, i] >= t).astype(int)
        f1_i = f1_score(val_true[:, i], preds_i, zero_division=0)
        if f1_i > best_f1:
            best_f1, best_t = f1_i, t
    best_thresholds[i] = best_t
    print(f"{label:15s} best_threshold={best_t:.2f}  Val F1={best_f1:.4f}")

# 3. Apply the tuned validation thresholds onto your new official Test Set
y_true_b, y_pred_baseline_test, y_prob_b = evaluate_model(model_bilstm, test_loader)

# Broadcast optimized thresholds matrix across the test probabilities
tuned_preds_b = (y_prob_b >= best_thresholds[None, :]).astype(int)

# Populate standard evaluation references for your Section 6 plots to read seamlessly
y_pred_b = tuned_preds_b 

print("\n================ FINAL TUNED REPORT ON SEPARATE TEST SET ================")
print(classification_report(y_true_b, tuned_preds_b, target_names=class_names, zero_division=0))

## 6. Plots: Loss, F1, Confusion Matrices & ROC Curves

In [ ]:
# ==========================================
# 1. Plot Loss & F1 History Curves (BiLSTM)
# ==========================================
actual_epochs = range(1, len(hist_bilstm["train_loss"]) + 1)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# BiLSTM Training Loss
axes[0].plot(actual_epochs, hist_bilstm["train_loss"], label="Train Loss", color="blue", marker='o')
axes[0].set_title("BiLSTM Training Performance (Loss)")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("BCE Loss")
axes[0].legend()
axes[0].grid(alpha=0.3)

# BiLSTM Validation F1 Progress
axes[1].plot(actual_epochs, hist_bilstm["val_f1"], label="Val F1", color="green", marker='o')
axes[1].axhline(0.75, color='red', linestyle='--', label='Target 0.75')
axes[1].set_title("Validation Weighted F1 per Epoch")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Weighted F1 Score")
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()


# ==========================================
# 2. Threshold Customization & Class Prevalence Charts
# ==========================================
# Compute per-label F1 metrics using default vs tuned arrays
default_per_label_f1 = f1_score(y_true_b, y_pred_baseline_test, average=None, zero_division=0)

# SAFEGUARD: Uses default predictions if you haven't run threshold tuning yet
if 'tuned_preds_b' not in locals():
    tuned_preds_b = y_pred_b

tuned_per_label_f1 = f1_score(y_true_b, tuned_preds_b, average=None, zero_division=0) 

# FIX: Removed len() around num_classes integer
x_pos = np.arange(num_classes)
width = 0.35

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Subplot A: Per-label F1 comparison
axes[0].bar(x_pos - width/2, default_per_label_f1, width, label='Threshold = 0.5', color='royalblue')
axes[0].bar(x_pos + width/2, tuned_per_label_f1, width, label='Tuned Threshold', color='darkorange')
axes[0].set_xticks(x_pos)
axes[0].set_xticklabels(class_names, rotation=30)
axes[0].set_ylabel('F1 Score')
axes[0].set_title('Per-label F1: Default vs Tuned Threshold')
axes[0].legend()
axes[0].grid(alpha=0.3, axis='y')

# Subplot B: Dataset imbalance reference chart
# FIX: Replaced undefined train_labels with train_df sum extraction
label_counts = train_df[LABEL_COLS].values.sum(axis=0) 
axes[1].bar(class_names, label_counts, color='steelblue')
axes[1].set_ylabel('Positive Count (Log Scale)')
axes[1].set_title('Label Prevalence in Training Set')
axes[1].set_yscale('log')
axes[1].set_xticklabels(class_names, rotation=30)
axes[1].grid(alpha=0.3, axis='y')

plt.tight_layout()
plt.show()


# ==========================================
# 3. BiLSTM Per-Class Confusion Matrices
# ==========================================
mcm_b = multilabel_confusion_matrix(y_true_b, y_pred_b)
fig, axes = plt.subplots(1, num_classes, figsize=(3.5 * num_classes, 3.5))

if num_classes == 1:
    axes = [axes]

for i, name in enumerate(class_names):
    sns.heatmap(mcm_b[i], annot=True, fmt="d", cmap="Blues", cbar=False, ax=axes[i])
    axes[i].set_title(f"BiLSTM Matrix: {name}")
    axes[i].set_xlabel("Predicted")
    axes[i].set_ylabel("Actual")
    
plt.tight_layout()
plt.show()


# ==========================================
# 4. BiLSTM Receiver Operating Characteristic (ROC) Curves
# ==========================================
fig, ax = plt.subplots(figsize=(8, 6))

for i in range(num_classes):
    fpr, tpr, _ = roc_curve(y_true_b[:, i], y_prob_b[:, i])
    ax.plot(fpr, tpr, label=f"{class_names[i]} (AUC = {auc(fpr, tpr):.2f})")

ax.plot([0, 1], [0, 1], 'k--')
ax.set_title("BiLSTM Multi-label ROC Curves (Per Class)")
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.legend(loc="lower right")
ax.grid(alpha=0.2)

plt.tight_layout()
plt.show()

In [ ]:
import torch
from transformers import BertTokenizer

# Save your BiLSTM model weights
torch.save(model_bilstm.state_dict(), "bilstm_model.pth")

# Save the BERT tokenizer so you can use it during inference later
bert_tokenizer.save_pretrained("./bert_tokenizer/")